# 02 · Modeling

**Goal:** fit several classifiers on 2022–2023 and leave 2024 for the comparison.

1. Load training data
2. GLM: a formula for the chance of a claim
3. Read the GLM coefficients
4. Five other classifiers, in plain language
5. Notes on modeling choices

In [28]:
import importlib
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))   # make `src` importable from notebooks/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import config
import src.train as train_mod

importlib.reload(config)
importlib.reload(train_mod)
fit_glm, fit_all, MODEL_NOTES = train_mod.fit_glm, train_mod.fit_all, train_mod.MODEL_NOTES

In [29]:
train = pd.read_csv(config.TRAIN_FILE)
print(train.shape, f"claim rate = {train[config.TARGET].mean():.2%}")

(185720, 18) claim rate = 13.64%


## 2. GLM

Why the **cloglog link with a log(exposure) offset**? If claims arrive as a Poisson process with rate λ, the probability of at least one claim over exposure *t* is 1 − exp(−λt). Taking cloglog of that gives log(λ) + log(t), so log(exposure) enters as an offset. TODO: explain this in your own words in the report.

In [30]:
glm = fit_glm(train)
glm.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                 Generalized Linear Model Regression Results                  
==============================================================================
Dep. Variable:                  claim   No. Observations:               185720
Model:                            GLM   Df Residuals:                   185675
Model Family:                Binomial   Df Model:                           44
Link Function:                CLogLog   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -68862.
Date:                Mon, 23 Nov 2026   Deviance:                   1.3772e+05
Time:                        02:35:00   Pearson chi2:                 1.96e+05
No. Iterations:                     6   Pseudo R-squ. (CS):            0.01798
Covariance Type:            nonrobust                                         
==================================================================================================
                                     coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept                         -1.3932      0.088    -15.880      0.000      -1.565      -1.221
C(bonus_score)[T.G]               -0.3620      0.040     -9.099      0.000      -0.440      -0.284
C(bonus_score)[T.N]               -0.2087      0.055     -3.804      0.000      -0.316      -0.101
C(policy_type)[T.COMP_E]           0.3666      0.015     24.100      0.000       0.337       0.396
C(policy_type)[T.COMP_N]           0.7937      0.023     34.113      0.000       0.748       0.839
C(policy_type)[T.TP]              -0.5975      0.080     -7.475      0.000      -0.754      -0.441
C(policy_type)[T.TPG]             -0.0733      0.024     -3.103      0.002      -0.120      -0.027
C(business_type)[T.P]              0.0190      0.015      1.294      0.196      -0.010       0.048
C(payment_frequency)[T.Q]          0.4310      0.025     17.343      0.000       0.382       0.480
C(payment_frequency)[T.S]          0.1980      0.017     11.586      0.000       0.165       0.232
C(fuel_type)[T.G]                 -0.1451      0.016     -8.865      0.000      -0.177      -0.113
C(fuel_type)[T.Unknown]            0.1428      0.097      1.468      0.142      -0.048       0.333
C(vehicle_brand)[T.BMW]            0.0370      0.038      0.966      0.334      -0.038       0.112
C(vehicle_brand)[T.CHEVROLET]     -0.1850      0.070     -2.642      0.008      -0.322      -0.048
C(vehicle_brand)[T.CITROEN]       -0.0624      0.036     -1.712      0.087      -0.134       0.009
C(vehicle_brand)[T.DACIA]          0.0712      0.089      0.804      0.422      -0.102       0.245
C(vehicle_brand)[T.DAEWOO]         0.0920      0.089      1.035      0.301      -0.082       0.266
C(vehicle_brand)[T.FIAT]          -0.0512      0.056     -0.916      0.360      -0.161       0.058
C(vehicle_brand)[T.FORD]          -0.0420      0.037     -1.134      0.257      -0.115       0.031
C(vehicle_brand)[T.HONDA]          0.0317      0.058      0.542      0.588      -0.083       0.146
C(vehicle_brand)[T.HYUNDAI]        0.0300      0.048      0.621      0.534      -0.065       0.125
C(vehicle_brand)[T.KIA]            0.0243      0.055      0.443      0.658      -0.083       0.132
C(vehicle_brand)[T.MAZDA]         -0.0172      0.069     -0.250      0.802      -0.152       0.117
C(vehicle_brand)[T.MERCEDES]      -0.0527      0.039     -1.345      0.178      -0.129       0.024
C(vehicle_brand)[T.MITSUBISHI]    -0.1343      0.074     -1.819      0.069      -0.279       0.010
C(vehicle_brand)[T.NISSAN]        -0.1459      0.045     -3.254      0.001      -0.234      -0.058
C(vehicle_brand)[T.OPEL]          -0.0450      0.037     -1.201      0.230      -0.118       0.028
C(vehicle_brand)[T.Other]         -0.1100      0.040     -2.769      0.006      -0.188      -0.032
C(vehicle_bran

## 3. Relativities

exp(β) is the multiplicative effect on the claim rate. Example: 1.25 means 25% higher risk than the reference level, all else equal.

In [31]:
relativities = pd.DataFrame({
    "relativity": np.exp(glm.params),
    "ci_low": np.exp(glm.conf_int()[0]),
    "ci_high": np.exp(glm.conf_int()[1]),
    "p_value": glm.pvalues,
}).round(3)
relativities

,relativity,ci_low,ci_high,p_value
Intercept,0.248,0.209,0.295,0.000
C(bonus_score)[T.G],0.696,0.644,0.753,0.000
C(bonus_score)[T.N],0.812,0.729,0.904,0.000
C(policy_type)[T.COMP_E],1.443,1.400,1.486,0.000
C(policy_type)[T.COMP_N],2.212,2.113,2.315,0.000
C(policy_type)[T.TP],0.550,0.470,0.644,0.000
C(policy_type)[T.TPG],0.929,0.887,0.973,0.002
C(business_type)[T.P],1.019,0.990,1.049,0.196
C(payment_frequency)[T.Q],1.539,1.466,1.616,0.000
C(payment_frequency)[T.S],1.219,1.179,1.261,0.000


**Interpretation:**

These sentences are for the first GLM, which used only driver age, vehicle age, and vehicle value. The saved model now also uses licence years, the car, the coverage, and where the car is driven.

Vehicle value matters most. From the 25th to the 75th percentile of declared value (about 14,000), the claim rate is multiplied by about 1.16, holding age fixed. Vehicle age is next: an 18-year gap, the same kind of middle spread, multiplies the rate by about 0.82, so older vehicles claim less often. Driver age has a relativity of about 1.00 (p = 0.99). A 38-year gap from the 5th to the 95th percentile barely moves the rate.

The flat driver-age result is the surprise. Driver age and vehicle age move together in this portfolio (correlation 0.86), so once vehicle age is in the model the linear driver-age term has nothing left to explain. A straight line also cannot pick up a young-driver spike. All three factors together still leave most claims unexplained (Cox–Snell pseudo R² about 0.005).

## 4. Other classifiers

Each model below answers the same question as the GLM: the probability of at least one claim. The predictors are the driver, the car, the coverage, where the car is driven, and exposure. Settings were fixed before looking at 2024.

- **Logistic regression.** One straight line for the log-odds of a claim. The inputs are scaled so vehicle value does not drown the ages.
- **Decision tree.** A few yes-or-no questions. Policies that take the same path share one claim rate. The tree stops after four questions.
- **Random forest.** Many trees, each trained on a random sample of policies. The probability is the average of the trees.
- **XGBoost.** Trees built in sequence. Each new tree corrects the policies the earlier trees got wrong.
- **LightGBM.** The same idea as XGBoost, with a different way of finding the splits.

In [32]:
models = fit_all(train)
test = train_mod.usable_rows(pd.read_csv(config.TEST_FILE))
scores = train_mod.score_models(models, test).rename(index=train_mod.DISPLAY_NAMES)
train_mod.save_comparison(scores)

results = scores.rename(columns={
    "AUC": "Sorting score (AUC)",
    "Gini": "Gini",
    "Brier": "Brier score",
    "calibration_gap": "Calibration gap",
    "mean_predicted": "Average predicted chance",
}).sort_values("Sorting score (AUC)", ascending=False).round(3)

for model, row in results.iterrows():
    print(
        f"{model}: sorting score {row['Sorting score (AUC)']:.3f}, "
        f"Gini {row['Gini']:.3f}, "
        f"calibration gap {row['Calibration gap']:.3f}"
    )
results

LightGBM: sorting score 0.692, Gini 0.384, calibration gap 0.003
XGBoost: sorting score 0.683, Gini 0.366, calibration gap 0.003
Random forest: sorting score 0.679, Gini 0.359, calibration gap 0.019
GLM: sorting score 0.679, Gini 0.359, calibration gap 0.005
Logistic regression: sorting score 0.676, Gini 0.352, calibration gap 0.008
Decision tree: sorting score 0.663, Gini 0.326, calibration gap 0.005


,Sorting score (AUC),Gini,Brier score,Calibration gap,Average predicted chance
LightGBM,0.692,0.384,0.120,0.003,0.150
XGBoost,0.683,0.366,0.121,0.003,0.150
Random forest,0.679,0.359,0.122,0.019,0.149
GLM,0.679,0.359,0.121,0.005,0.149
Logistic regression,0.676,0.352,0.122,0.008,0.150
Decision tree,0.663,0.326,0.122,0.005,0.149


## 4b. Comparison on 2024

Each model learned from 2022 and 2023, then was tested on 2024. A taller bar means it is better at putting drivers who filed a claim above drivers who did not. LightGBM is the tallest.

![Six classifiers compared on 2024](../reports/figures/model_comparison.png)

## 5. Modeling choices

Every model estimates the probability of at least one claim. About 14% of training policies have a claim, and that frequency is the base rate the probabilities should match. Oversampling or class weights would push the predictions toward a 50/50 problem, so they would no longer be the chance of a claim. A policy that was only in force for part of the year is handled in the GLM by a log(exposure) offset. The other models receive exposure as a column.

The tree settings were fixed before seeing 2024: the single tree is four questions deep, the forest has 200 trees, and XGBoost and LightGBM each have 300 trees with a learning rate of 0.05. There is no cross-validated search. Any later search has to stay inside 2022–2023.

The first GLM formula named `years_licensed` and `vehicle_type`. Those columns are not in the training file, so the fit failed until the formula used `driver_age`, `vehicle_age`, and `vehicle_value`. A linear driver-age term then added nothing once vehicle age was included. Rows with a missing predictor or zero exposure are left out of the GLM, because the offset is log(exposure).